<!--nav--> [🗺 Learning path](README.md) · **33/40** · ◀ [The Hardware Roofline: NVIDIA vs AMD](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) · [The What-If Console](./Serving_WhatIf_Console.ipynb) ▶

# Portable Kernels & Precision: CUDA, HIP, Triton — and What Actually Runs Where

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Portable_Kernels_Precision_Matrix.ipynb)

Notebook 31 showed the *physics* is the same on both vendors. This one is about the part that
isn't: **which optimization actually exists, on which architecture, in which framework, today.**

This is where serving projects die. Someone benchmarks AWQ on an A100, ships the config to an
MI300X fleet, and discovers the fast int4 kernel they were relying on is NVIDIA-only. Or they
enable FP8 on an A100 — which has no FP8 hardware at all — and silently get emulation.

| Part | What you'll learn |
|---|---|
| **1** | The porting stack: CUDA → HIP → Triton, and what each layer costs you |
| **2** | **The precision support matrix** — every dtype × every architecture, as data + a D3 heatmap |
| **3** | **The kernel matrix** — FlashAttention, paged attention, int4 GEMM, MoE, collectives |
| **4** | Writing **one kernel that runs on both vendors** (a real Triton kernel, executed if you have a GPU) |
| **5** | The vLLM flag translation table: NVIDIA ↔ AMD |
| **6** | A portability checklist and the failure modes to expect |

**Runs on:** any CPU for Parts 1–3, 5–6. Part 4 runs the kernel on CUDA *or* ROCm if present.

> ⚠️ **This is the fastest-moving content in the track.** Kernel support changes monthly — AMD's
> AITER, vLLM's backends, and FP4/MXFP4 support are all actively landing. Treat the matrices as a
> *snapshot and a way of thinking*, and re-verify against release notes before you commit to a
> hardware plan. Every claim below is one you should check with the probe in Part 4 on your own box.

In [ ]:
import json, uuid
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · The porting stack

There are four levels at which you can write GPU code, and they trade **portability against peak
performance** in a very predictable way:

```
 ┌─────────────────────────────────────────────────────────────────────────┐
 │ LEVEL 4  Framework ops        torch.matmul, F.scaled_dot_product_attention │  100% portable
 │          "someone else's problem"                                        │  ~70-95% of peak
 ├─────────────────────────────────────────────────────────────────────────┤
 │ LEVEL 3  Triton / torch.compile                                          │  ~portable
 │          one kernel, both vendors, JIT-tuned per arch                    │  ~80-95% of peak
 ├─────────────────────────────────────────────────────────────────────────┤
 │ LEVEL 2  Vendor template libs   CUTLASS (NV)  /  Composable Kernel (AMD) │  per-vendor source
 │                                                                          │  ~95-100% of peak
 ├─────────────────────────────────────────────────────────────────────────┤
 │ LEVEL 1  Hand-written intrinsics  mma.sync (NV) / v_mfma (AMD), asm      │  vendor+arch locked
 │          FlashAttention-3, Marlin, AITER kernels live here               │  100% of peak
 └─────────────────────────────────────────────────────────────────────────┘
```

**HIP sits beside CUDA at levels 1–2, not above them.** `hipify` mechanically rewrites CUDA source
to HIP (`cudaMalloc`→`hipMalloc`, `__shfl_sync`→`__shfl`), and it works: the code compiles and runs.
What it *cannot* do is retune it. A kernel whose tile sizes, register budget, and reduction tree
were chosen for **32-lane warps** will be correct but mediocre on **64-lane wavefronts** (nb 31).

That gap — "ported" vs "tuned" — is the single best predictor of whether a given optimization is
fast on both vendors today:

| Optimization | Why it's fast on NVIDIA | State on AMD |
|---|---|---|
| Dense GEMM | cuBLASLt, decades of tuning | hipBLASLt — mature, competitive |
| Attention | FlashAttention-2/3, hand-written for Hopper | CK-based FA + Triton FA; **AITER** for inference |
| int4 weight GEMM | **Marlin/Machete** — Ampere+ `mma` intrinsics | different kernels; historically the biggest gap |
| Collectives | NCCL | RCCL (NCCL API port) — mature |
| Graph capture | CUDA Graphs | HIP Graphs — supported |

**The practical rule:** the closer an optimization sits to level 1, the more likely it is
vendor-specific *in practice*, even when the hardware supports the concept on both sides.

## Part 2 · The precision support matrix

"Does this GPU support FP8?" has three different answers depending on what you mean:

- **hardware**: are there instructions for it?
- **framework**: does PyTorch/vLLM expose it?
- **fast path**: is there a *tuned kernel*, or does it fall back to emulation?

The third is the one that matters and the one nobody publishes. Here's the matrix as data:

In [ ]:
# 3 = native + tuned kernels, 2 = works but not fully optimized, 1 = emulated/slow, 0 = unsupported
ARCHS = [
    ("Turing (T4)",        "NVIDIA"),
    ("Ampere (A100/A10)",  "NVIDIA"),
    ("Ada (L4/L40S)",      "NVIDIA"),
    ("Hopper (H100/H200)", "NVIDIA"),
    ("Blackwell (B200)",   "NVIDIA"),
    ("CDNA2 (MI210/250X)", "AMD"),
    ("CDNA3 (MI300X/325X)","AMD"),
    ("CDNA4 (MI355X)",     "AMD"),
]

PRECISIONS = ["fp32", "tf32", "fp16", "bf16", "fp8 (E4M3)", "int8", "int4 (AWQ/GPTQ)", "fp4/mxfp4"]

# rows = precision, cols = arch (same order as ARCHS)
SUPPORT = {
  "fp32":            [3, 3, 3, 3, 3, 3, 3, 3],
  "tf32":            [0, 3, 3, 3, 3, 0, 0, 0],   # tf32 is an NVIDIA tensor-core format
  "fp16":            [3, 3, 3, 3, 3, 3, 3, 3],
  "bf16":            [0, 3, 3, 3, 3, 3, 3, 3],   # Turing has no bf16
  "fp8 (E4M3)":      [0, 0, 3, 3, 3, 0, 3, 3],   # Ada/Hopper/Blackwell; CDNA3+
  "int8":            [3, 3, 3, 3, 3, 2, 3, 3],
  "int4 (AWQ/GPTQ)": [2, 3, 3, 3, 3, 1, 2, 2],   # Marlin needs Ampere+; AMD path less mature
  "fp4/mxfp4":       [0, 0, 0, 0, 3, 0, 0, 3],   # Blackwell / CDNA4
}
LEGEND = {3: "native + tuned", 2: "works, not fully tuned", 1: "emulated / slow", 0: "unsupported"}

print(f"{'precision':<18}" + "".join(f"{a[0][:11]:<13}" for a in ARCHS))
print("-" * 122)
SYM = {3: "  ███ ", 2: "  ▓▓  ", 1: "  ░   ", 0: "   ·  "}
for p in PRECISIONS:
    print(f"{p:<18}" + "".join(f"{SYM[v]:<13}" for v in SUPPORT[p]))
print("\nlegend: ███ native+tuned   ▓▓ works but not fully tuned   ░ emulated/slow   · unsupported")

print("\nThe three rows that decide real deployments:")
print("  bf16  - Turing (T4) has NONE. Colab's free GPU cannot run bf16 checkpoints natively;")
print("          that is why every notebook in this track passes dtype='half' on T4.")
print("  fp8   - Ada/Hopper/Blackwell and CDNA3+. On A100 there is NO fp8 hardware, so")
print("          '--quantization fp8' there means emulation, not speed.")
print("  int4  - the widest vendor gap. Marlin-class kernels are Ampere+ NVIDIA;")
print("          plan to re-benchmark rather than assume portability of nb 23's results.")

In [ ]:
# Heatmap: precision x architecture, colored by maturity.
cells = []
for pi, p in enumerate(PRECISIONS):
    for ai, (arch, vendor) in enumerate(ARCHS):
        cells.append({"p": p, "arch": arch, "vendor": vendor,
                      "pi": pi, "ai": ai, "v": SUPPORT[p][ai]})

JS = r'''
const M = {top: 74, right: 20, bottom: 20, left: 132};
const archs = [...new Set(data.cells.map(d => d.arch))];
const precs = [...new Set(data.cells.map(d => d.p))];
const iw = W - M.left - M.right;
const cw = iw / archs.length, ch = 34;
const svg = root.append("svg").attr("width", W).attr("height", M.top + precs.length*ch + M.bottom)
    .append("g").attr("transform",`translate(${M.left},${M.top})`);
const color = d3.scaleOrdinal().domain([0,1,2,3])
    .range(["#eceff1", "#ffe0b2", "#ffca28", "#43a047"]);

svg.selectAll("c").data(data.cells).join("rect")
   .attr("x", d => d.ai*cw + 2).attr("y", d => d.pi*ch + 2)
   .attr("width", cw-4).attr("height", ch-4).attr("rx",4)
   .attr("fill", d => color(d.v))
   .append("title").text(d => `${d.p} on ${d.arch}: ${data.legend[d.v]}`);
svg.selectAll("t").data(data.cells).join("text")
   .attr("x", d => d.ai*cw + cw/2).attr("y", d => d.pi*ch + ch/2 + 4)
   .attr("text-anchor","middle").style("font-size","10.5px")
   .style("fill", d => d.v === 3 ? "white" : "#455a64")
   .text(d => ["—","emu","ok","fast"][d.v]);
svg.selectAll("rl").data(precs).join("text")
   .attr("x",-8).attr("y",(d,i)=>i*ch+ch/2+4).attr("text-anchor","end")
   .style("font-size","11.5px").style("font-family","ui-monospace,monospace").text(d=>d);
svg.selectAll("cl").data(archs).join("text")
   .attr("x",(d,i)=>i*cw+cw/2).attr("y",-8).attr("text-anchor","start")
   .attr("transform",(d,i)=>`rotate(-38 ${i*cw+cw/2} -8)`)
   .style("font-size","10.5px")
   .style("fill",(d,i)=> data.cells.find(c=>c.arch===d).vendor === "NVIDIA" ? "#76b900" : "#ed1c24")
   .text(d=>d);
'''
show_d3(JS, {"cells": cells, "legend": LEGEND}, height=380)

## Part 3 · The kernel matrix

Precision support is necessary but not sufficient — you also need the *kernel* that uses it. Here
are the kernels every serving deployment depends on, and where each one comes from:

| Capability | NVIDIA path | AMD path | Portability note |
|---|---|---|---|
| **Attention (prefill)** | FlashAttention-2/3 (FA-3 is Hopper-specific) | CK Flash Attention, Triton FA, **AITER** | concept portable; the *fastest* implementation is always vendor-specific |
| **Paged attention (decode)** | vLLM CUDA kernels | vLLM ROCm kernels / AITER | both supported in vLLM |
| **Dense GEMM** | cuBLASLt, CUTLASS | hipBLASLt, CK | mature both sides |
| **int4 weight-only GEMM** | **Marlin / Machete** (Ampere+) | ROCm int4 paths, less mature | ⚠️ biggest gap — re-benchmark, never assume |
| **FP8 GEMM** | cuBLASLt FP8 (Ada/Hopper+) | hipBLASLt FP8 (CDNA3+) | both native where hardware exists |
| **MoE dispatch** | CUTLASS grouped GEMM | CK grouped GEMM / AITER | both work; perf varies by expert count |
| **Collectives** | NCCL | RCCL | API-compatible |
| **Graph capture** | CUDA Graphs | HIP Graphs | vLLM V1 uses this on both |
| **Sampling / logits** | vLLM CUDA | Triton (portable) | Triton makes this a non-issue |

**Where Triton fits.** OpenAI Triton has a mature AMD backend. That makes it the *pragmatic*
portability layer for everything except the top handful of hot kernels: you write one kernel, and
the compiler picks lane counts, tiles, and scheduling per architecture. vLLM leans on this heavily —
which is exactly why vLLM runs on ROCm at all rather than being a CUDA-only project.

## Part 4 · One kernel, both vendors

Enough theory. Here's a real Triton kernel — a fused RMSNorm, which every modern LLM runs twice per
layer. The **same source** compiles and runs on CUDA and ROCm.

In [ ]:
# A portable Triton kernel. Runs on NVIDIA or AMD; falls back to an explanation on CPU.
import torch

try:
    import triton
    import triton.language as tl
    HAVE_TRITON = True
except ImportError:
    HAVE_TRITON = False

SRC = '''
@triton.jit
def rmsnorm_kernel(X, W, Y, stride, N, EPS: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    X += row * stride
    Y += row * stride
    # 1) sum of squares, accumulated in fp32 regardless of input dtype
    acc = tl.zeros([BLOCK], dtype=tl.float32)
    for off in range(0, N, BLOCK):
        idx = off + tl.arange(0, BLOCK)
        x = tl.load(X + idx, mask=idx < N, other=0.0).to(tl.float32)
        acc += x * x
    rms = tl.sqrt(tl.sum(acc) / N + EPS)
    # 2) normalize and scale
    for off in range(0, N, BLOCK):
        idx = off + tl.arange(0, BLOCK)
        x = tl.load(X + idx, mask=idx < N, other=0.0).to(tl.float32)
        w = tl.load(W + idx, mask=idx < N, other=0.0).to(tl.float32)
        tl.store(Y + idx, (x / rms * w).to(Y.dtype.element_ty), mask=idx < N)
'''
print("The kernel (identical source on both vendors):")
print(SRC)
print("Note what is NOT in it: no warp size, no wavefront size, no vendor intrinsics.")
print("Triton's compiler chooses those per architecture - that is the whole portability argument.")

In [ ]:
# Execute + benchmark it against PyTorch, on whatever GPU is present.
if not torch.cuda.is_available():
    print("No GPU here - skipping execution.")
    print("On a CUDA or ROCm machine this cell compiles the kernel above and benchmarks it.")
elif not HAVE_TRITON:
    print("Triton not installed (pip install triton) - skipping execution.")
else:
    import time
    hip = getattr(torch.version, "hip", None)
    print(f"vendor: {'AMD (ROCm ' + hip + ')' if hip else 'NVIDIA (CUDA ' + str(torch.version.cuda) + ')'}")
    print(f"device: {torch.cuda.get_device_name(0)}\n")

    exec(compile("import triton\nimport triton.language as tl\n" + SRC, "<rmsnorm>", "exec"), globals())

    def rmsnorm_triton(x, w, eps=1e-6):
        y = torch.empty_like(x)
        M, N = x.shape
        BLOCK = 1024
        rmsnorm_kernel[(M,)](x, w, y, x.stride(0), N, EPS=eps, BLOCK=BLOCK)
        return y

    def rmsnorm_torch(x, w, eps=1e-6):
        v = x.float()
        return (v / torch.sqrt(v.pow(2).mean(-1, keepdim=True) + eps) * w.float()).to(x.dtype)

    M, N = 4096, 4096
    x = torch.randn(M, N, device="cuda", dtype=torch.float16)
    w = torch.randn(N, device="cuda", dtype=torch.float16)

    out_t, out_r = rmsnorm_triton(x, w), rmsnorm_torch(x, w)
    err = (out_t.float() - out_r.float()).abs().max().item()
    print(f"max abs difference vs PyTorch reference: {err:.2e}  "
          f"({'✓ correct' if err < 1e-2 else '✗ MISMATCH'})")

    def bench(fn, iters=100):
        for _ in range(10): fn()
        torch.cuda.synchronize(); t0 = time.perf_counter()
        for _ in range(iters): fn()
        torch.cuda.synchronize()
        return (time.perf_counter() - t0) / iters * 1e6      # microseconds

    t_tri = bench(lambda: rmsnorm_triton(x, w))
    t_pt  = bench(lambda: rmsnorm_torch(x, w))
    gb = (2 * x.numel() * x.element_size()) / 1e9            # read + write
    print(f"\n{'impl':<22}{'time':>10}{'effective BW':>16}")
    print(f"{'Triton (portable)':<22}{t_tri:>8.1f}us{gb/(t_tri/1e6):>14.0f} GB/s")
    print(f"{'PyTorch eager':<22}{t_pt:>8.1f}us{gb/(t_pt/1e6):>14.0f} GB/s")
    print(f"\nspeedup: {t_pt/t_tri:.2f}x  (fusion avoids materializing intermediates)")
    print("Compare the effective bandwidth to your measured peak from notebook 31 -")
    print("a good elementwise kernel should approach it, because it IS memory-bound.")

**Why this matters beyond the microbenchmark.** RMSNorm is memory-bound (nb 31: AI ≈ 1 FLOP/byte).
The eager PyTorch version materializes several intermediate tensors, each a full read+write of HBM.
The fused kernel touches memory twice: once in, once out. That's the entire speedup, and it's the
same story on both vendors — **because the bottleneck is physics, not vendor**.

## Part 5 · vLLM flags: NVIDIA ↔ AMD translation

Most vLLM flags are identical across vendors. These are the ones that aren't, or that behave
differently:

| Goal | NVIDIA | AMD (ROCm) |
|---|---|---|
| Install | `pip install vllm` | ROCm wheel / official ROCm Docker image |
| Attention backend | `VLLM_ATTENTION_BACKEND=FLASH_ATTN` (default) | `VLLM_USE_TRITON_FLASH_ATTN=1`, or AITER via `VLLM_ROCM_USE_AITER=1` |
| int4 weights | `--quantization awq_marlin` / `gptq_marlin` (Ampere+) | `--quantization awq` / `gptq` — verify the kernel path is fast |
| FP8 | `--quantization fp8` (Ada/Hopper+) | `--quantization fp8` (CDNA3+) |
| FP8 KV cache | `--kv-cache-dtype fp8` (Ada/Hopper+) | `--kv-cache-dtype fp8` (CDNA3+) |
| Multi-GPU | `--tensor-parallel-size N` (NCCL) | same flag (RCCL) |
| Topology check | `nvidia-smi topo -m` | `rocm-smi --showtopo` |
| Monitoring | `nvidia-smi`, DCGM | `rocm-smi`, `amd-smi` |
| Graph capture | on by default | on by default |

**The trap that costs people weeks:** a flag being *accepted* is not a flag being *fast*. `awq` on a
card with no tuned int4 kernel will run — slowly. Always confirm with a benchmark (notebook 23's
harness works unchanged on ROCm), never with the absence of an error message.

Detect what your own box actually supports:

In [ ]:
# A capability probe you can drop into any deployment script - vendor-agnostic.
import torch

def probe():
    if not torch.cuda.is_available():
        return {"gpu": None}
    p = torch.cuda.get_device_properties(0)
    hip = getattr(torch.version, "hip", None)
    vendor = "AMD" if hip else "NVIDIA"
    cc = None if hip else (p.major, p.minor)
    caps = {
        "vendor": vendor,
        "device": p.name,
        "vram_gb": round(p.total_memory / 1e9, 1),
        "runtime": hip or torch.version.cuda,
        "compute_capability": f"{cc[0]}.{cc[1]}" if cc else f"gfx (CDNA/RDNA: {getattr(p, 'gcnArchName', 'unknown')})",
        "bf16": torch.cuda.is_bf16_supported(),
        # fp8 dtypes exist in torch; hardware support is a separate question:
        "fp8_dtype_in_torch": hasattr(torch, "float8_e4m3fn"),
        "fp8_hardware_likely": (cc is not None and cc[0] >= 9) or (cc is not None and cc == (8, 9))
                               or (hip is not None and "gfx94" in str(getattr(p, "gcnArchName", ""))),
        "flash_sdp": torch.backends.cuda.flash_sdp_enabled(),
        "mem_efficient_sdp": torch.backends.cuda.mem_efficient_sdp_enabled(),
    }
    try:
        import triton; caps["triton"] = triton.__version__
    except ImportError:
        caps["triton"] = None
    return caps

info = probe()
if not info.get("vendor"):
    print("No GPU - here is what the probe returns on a real machine:")
    print(json.dumps({"vendor": "AMD", "device": "AMD Instinct MI300X", "vram_gb": 192.0,
                      "runtime": "6.2.41133", "compute_capability": "gfx942",
                      "bf16": True, "fp8_dtype_in_torch": True, "fp8_hardware_likely": True,
                      "flash_sdp": True, "mem_efficient_sdp": True, "triton": "3.x"}, indent=2))
else:
    print(json.dumps(info, indent=2))
    print("\nUse this in CI: fail the deploy if a config asks for a precision the")
    print("hardware doesn't have, instead of discovering it as a 10x slowdown in production.")

## Part 6 · The portability checklist

Before you promise anyone that a serving config is vendor-portable:

- [ ] **Re-benchmark, don't re-reason.** Notebook 23's quantization harness and notebook 27's load
      generator both run unchanged on ROCm. Numbers, not assumptions.
- [ ] **Check the dtype is native, not emulated** — Part 5's probe, in CI.
- [ ] **Check bf16 before assuming it** (T4 has none; many checkpoints ship bf16 weights).
- [ ] **Re-tune batch size per GPU.** Different ridge points (nb 31) mean the same `--max-num-seqs`
      lands in a different regime.
- [ ] **Re-tune TP width per GPU.** A model needing TP=4 on 80 GB cards may need TP=1 on 192 GB
      cards — which changes routing, failure domains, and cost (nb 29, 31).
- [ ] **Recompute the KV pool.** Read the startup log on the new hardware (nb 26); it's the fastest
      capacity check you have.
- [ ] **Re-validate accuracy after switching quantization kernels.** Different kernels, different
      numerics — run notebook 23's quiz plus a real eval.
- [ ] **Watch for silent fallbacks in the logs.** "using Triton fallback", "kernel not available",
      "emulated" — notebook 26 taught you to read these.

### Expected failure modes, ranked by how often they bite

1. **Silent slow path** — everything works, throughput is 3× worse. Caught only by benchmarking.
2. **Missing quantized kernel** — the checkpoint loads, decode is slower than fp16.
3. **bf16 assumed** — crash or silent fp32 upcast on Turing.
4. **Different attention backend chosen** — different numerics; your golden outputs shift slightly.
5. **Collectives topology** — TP works but is slow because the GPUs aren't on the fast fabric.

## Recap

- **Four levels of GPU code**; portability and peak performance trade off predictably. Triton is the
  sweet spot for everything except a handful of hot kernels.
- **HIP makes CUDA code *run* on AMD; it does not make it *fast*** — warp 32 vs wavefront 64 is why
  vendors ship their own tuned libraries (CUTLASS/CK, FA-3/AITER).
- **Support has three levels** — hardware, framework, *tuned kernel*. Only the third one is speed.
- **The widest real gap is int4 weight-only GEMM**; FP8 is native on both modern vendors; bf16 is a
  Turing-era trap.
- **Probe capabilities in CI**, and re-benchmark on every new architecture rather than porting
  conclusions.

### Further reading
- [Triton](https://triton-lang.org/) · [AMD Composable Kernel](https://github.com/ROCm/composable_kernel) · [AITER](https://github.com/ROCm/aiter) · [CUTLASS](https://github.com/NVIDIA/cutlass)
- [vLLM on ROCm — installation & tuning](https://docs.vllm.ai/en/latest/getting_started/installation/gpu/index.html)
- [HIP porting guide](https://rocm.docs.amd.com/projects/HIP/en/latest/how-to/hip_porting_guide.html)
- [FlashAttention-2](https://arxiv.org/abs/2307.08691) · [FlashAttention-3](https://arxiv.org/abs/2407.08608) (Hopper-specific) · [Marlin](https://github.com/IST-DASLab/marlin)
- [OCP FP8 specification](https://arxiv.org/abs/2209.05433) — the format both vendors implement